In [1]:
#### ------------------------------------------------------------------------------------------
#### author: Ranjan Barman, date: July 24, 2025 (modified for POST_NAT_BRCA)
#### Mapped POST_NAT HoverNet Immune-Only NPIFs (top 25% tiles) to BRCA subtype status
#### --------------------------------------------------------------------------------------------

import os
import pandas as pd

# Set working directory
_wpath_ = "/data/Lab_ruppin/Ranjan/HnE/"
os.chdir(_wpath_)
print(f"Working directory: {_wpath_}\n")

# Dataset name
dataset_name = "POST_NAT_BRCA"

# File paths
npif_file = f"{dataset_name}/HoverNet/outputs/POST_NAT_BRCA_HoverNet_ImmuneOnly_NPIFs_Filtered_Tiles_Top25Q.csv"
slide_list_file = "/data/Ruppin_AI/Datasets/Post_NAT_BRCA/processed/Post_NAT_BRCA_slide_list.tsv"
clinical_metadata_file = "/data/Ruppin_AI/Datasets/Post_NAT_BRCA/processed/Post_NAT_BRCA_clinical_metadata_short.tsv"

# Load NPIFs and extract Slide_ID
npif_df = pd.read_csv(npif_file)
npif_df["Slide_ID"] = npif_df["Slide_ID"].astype(int)

# Load slide-to-patient mapping and clinical metadata
slide_list_df = pd.read_csv(slide_list_file, sep="\t")
clinical_metadata_df = pd.read_csv(clinical_metadata_file, sep="\t")
slide_list_df
clinical_metadata_df


Working directory: /data/Lab_ruppin/Ranjan/HnE/



,Patient_ID,Age,Menopausal_status,Lymphovascular_invasion,Histology_type,Histology_grade,HER2_status,ER_status,PR_status,Clinical_subtype,Clinical_subtype_fine,NAT_regimen,Surgery_type_breast,Surgery_type_LN,Response
0,P1,67,Post,1.0,IDC,3,0.0,1.0,1.0,HR+,HR+,Chemo+Endocrine,Total mastectomy,Axillary LN dissection,PDR
1,P2,30,Pre,1.0,IDC,3,1.0,0.0,0.0,HER2+,HER2+,Chemo+Anti-HER2,Total mastectomy,Axillary LN dissection,PDR
2,P3,65,Post,1.0,IDC,2,0.0,1.0,1.0,HR+,HR+,Chemo,Partial mastectomy/lumpectomy,Axillary LN dissection,PDR
3,P4,41,Pre,0.0,IDC,2,0.0,1.0,1.0,HR+,HR+,Chemo,Total mastectomy,Sentinel LN biopsy,NDR
4,P5,47,Pre,0.0,IDC,Undetermined,1.0,1.0,1.0,HER2+,TPBC,Chemo+Anti-HER2,Total mastectomy,Axillary LN dissection,PDR
5,P6,58,Post,1.0,ILC,2,0.0,1.0,1.0,HR+,HR+,Chemo,Total mastectomy,Axillary LN dissection,PDR
6,P7,39,Pre,0.0,IDC,3,0.0,0.0,0.0,TNBC,TNBC,Chemo,Total mastectomy,Sentinel LN biopsy,PDR
7,P8,52,Pre,0.0,IDC,2,0.0,1.0,1.0,HR+,HR+,Chemo,Total mastectomy,Axillary LN dissection,PDR
8,P9,34,Pre,0.0,IDC,2,1.0,1.0,0.0,HER2+,TPBC,Chemo+Anti-HER2,Total mastectomy,Axillary LN dissection,PDR
9,P10,55,Post,1.0,IDC,2,0.0,1.0,1.0,HR+,HR+,Chemo,Total mastectomy,Axillary LN dissection,PDR


In [2]:
# Merge Slide_ID to get Patient_ID
npif_mapped_df = pd.merge(npif_df, slide_list_df[["Patient_ID", "Slide_ID"]], on="Slide_ID", how="left")
npif_mapped_df = npif_mapped_df.dropna(subset=["Patient_ID"])
npif_mapped_df["Patient_ID"] = npif_mapped_df["Patient_ID"].astype(str)

# Drop any pre-existing clinical columns to avoid duplication
columns_to_drop = ["HER2_Status", "ER_Status", "PR_Status", "Clinical_subtype"]
npif_mapped_df = npif_mapped_df.drop(columns=[col for col in columns_to_drop if col in npif_mapped_df.columns])

# Prepare and rename clinical subtype columns
clinical_metadata_df["Patient_ID"] = clinical_metadata_df["Patient_ID"].astype(str)
subtypes_df = clinical_metadata_df.rename(columns={
    "HER2_status": "HER2_Status",
    "ER_status": "ER_Status",
    "PR_status": "PR_Status"
})[["Patient_ID", "HER2_Status", "ER_Status", "PR_Status", "Clinical_subtype"]]

# Map binary values to 'Positive'/'Negative'
binary_map = {1.0: "Positive", 0.0: "Negative"}
subtypes_df["HER2_Status"] = subtypes_df["HER2_Status"].map(binary_map)
subtypes_df["ER_Status"] = subtypes_df["ER_Status"].map(binary_map)
subtypes_df["PR_Status"] = subtypes_df["PR_Status"].map(binary_map)

# Merge subtype info into npif_mapped_df
merged_df = pd.merge(subtypes_df, npif_mapped_df, on="Patient_ID", how="inner")

# Reorder to make Patient_ID the first column
cols = merged_df.columns.tolist()
cols.insert(0, cols.pop(cols.index("Patient_ID")))
merged_df = merged_df[cols]

# Print unique Slide_Name and Patient_ID values
print("Patient_IDs from POST_NAT_BRCA_HoverNet_NPIFs (Total: {}):".format(len(merged_df)))
print(merged_df["Patient_ID"].unique())

print("\nPatient IDs from POST_NAT clinical metadata (Total: {}):".format(len(subtypes_df)))
print(subtypes_df["Patient_ID"].unique())

# Find non-matching values
slide_ids = set(merged_df["Patient_ID"])
patient_ids = set(subtypes_df["Patient_ID"])

non_matching_slides = slide_ids - patient_ids
non_matching_patients = patient_ids - slide_ids

print("\nHoverNet NPIFs that do NOT have a matching Patient_ID:")
print(non_matching_slides)

print("\nClinical metadata entries that do NOT have a matching Slide NPIF:")
print(non_matching_patients)



Patient_IDs from POST_NAT_BRCA_HoverNet_NPIFs (Total: 93):
['P1' 'P2' 'P3' 'P4' 'P5' 'P6' 'P7' 'P8' 'P9' 'P10' 'P11' 'P12' 'P13'
 'P14' 'P15' 'P16' 'P17' 'P18' 'P19' 'P20' 'P21' 'P22' 'P23' 'P24' 'P25'
 'P26' 'P27' 'P28' 'P29' 'P30' 'P31' 'P32' 'P33' 'P34' 'P35' 'P37' 'P38'
 'P39' 'P40' 'P41' 'P42' 'P43' 'P44' 'P45' 'P46' 'P47' 'P48' 'P49' 'P50'
 'P51' 'P52' 'P53' 'P54']

Patient IDs from POST_NAT clinical metadata (Total: 54):
['P1' 'P2' 'P3' 'P4' 'P5' 'P6' 'P7' 'P8' 'P9' 'P10' 'P11' 'P12' 'P13'
 'P14' 'P15' 'P16' 'P17' 'P18' 'P19' 'P20' 'P21' 'P22' 'P23' 'P24' 'P25'
 'P26' 'P27' 'P28' 'P29' 'P30' 'P31' 'P32' 'P33' 'P34' 'P35' 'P36' 'P37'
 'P38' 'P39' 'P40' 'P41' 'P42' 'P43' 'P44' 'P45' 'P46' 'P47' 'P48' 'P49'
 'P50' 'P51' 'P52' 'P53' 'P54']

HoverNet NPIFs that do NOT have a matching Patient_ID:
set()

Clinical metadata entries that do NOT have a matching Slide NPIF:
{'P36'}


In [3]:
# Save merged (slide-level) result
output_dir = f"{dataset_name}/outputs/HoverNet/Subtypes/"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "HoverNet_ImmuneOnly_NPIFs_Filtered_Tiles_Top25Q_POST_NAT_BRCA_Mapped_BRCA_Status.csv")
# --------------------------------------
# Patient-level aggregation of NPIFs
# --------------------------------------

# Identify NPIF feature columns to average
mean_std_cols = [col for col in merged_df.columns if col.startswith("Mean ") or col.startswith("Std ")]

# Aggregate NPIFs by Patient_ID (mean), keep first for clinical/status columns
patient_level_df = merged_df.groupby("Patient_ID").agg({
    "HER2_Status": "first",
    "ER_Status": "first",
    "PR_Status": "first",
    "Clinical_subtype": "first",
    **{col: "mean" for col in mean_std_cols}
}).reset_index()

# Save patient-level aggregated results to the same file (overwrite)
patient_level_df.to_csv(output_file, index=False)
print(f"\nPatient-level averaged data saved to: {output_file}")
patient_level_df



Patient-level averaged data saved to: POST_NAT_BRCA/outputs/HoverNet/Subtypes/HoverNet_ImmuneOnly_NPIFs_Filtered_Tiles_Top25Q_POST_NAT_BRCA_Mapped_BRCA_Status.csv


,Patient_ID,HER2_Status,ER_Status,PR_Status,Clinical_subtype,Mean Area,Mean Major Axis,Mean Minor Axis,Mean Perimeter,Mean Eccentricity,Mean Circularity,Std Area,Std Major Axis,Std Minor Axis,Std Perimeter,Std Eccentricity,Std Circularity
0,P1,Negative,Positive,Positive,HR+,5.595238,3.229980,2.369887,9.262728,0.635549,0.804776,2.003458,0.653231,0.407477,1.725388,0.148471,0.068176
1,P10,Negative,Positive,Positive,HR+,5.676457,3.268964,2.370534,9.345202,0.643008,0.798900,2.251475,0.738794,0.437702,1.942310,0.147696,0.070763
2,P11,Negative,Negative,Negative,TNBC,6.421978,3.404499,2.559704,9.875950,0.615621,0.812667,2.166338,0.668359,0.414613,1.739786,0.147808,0.059766
3,P12,Negative,Negative,Negative,TNBC,6.320812,3.455951,2.485484,9.867470,0.646780,0.799920,2.357049,0.758925,0.435252,1.938232,0.152184,0.069328
4,P13,Negative,Negative,Negative,TNBC,8.718806,3.920496,2.911623,11.382068,0.632550,0.798511,4.456841,1.009659,0.758868,2.914714,0.143901,0.062679
5,P14,Negative,Positive,Positive,HR+,5.986627,3.367562,2.425403,9.613668,0.650251,0.797193,2.177529,0.703087,0.431777,1.844691,0.146488,0.067229
6,P15,Negative,Positive,Positive,HR+,6.184655,3.434709,2.439945,9.759147,0.661048,0.795443,2.503895,0.776890,0.474699,2.062157,0.144514,0.068737
7,P16,Negative,Positive,Positive,HR+,5.394142,3.184713,2.320837,9.107107,0.635108,0.804531,1.945574,0.692897,0.387248,1.777684,0.153343,0.070659
8,P17,Negative,Positive,Positive,HR+,5.746051,3.284166,2.384626,9.400563,0.642973,0.798126,2.228415,0.711554,0.453374,1.897446,0.148393,0.069909
9,P18,Negative,Negative,Negative,TNBC,5.352612,3.188911,2.302126,9.093483,0.645106,0.801647,1.824267,0.664038,0.378006,1.708493,0.150249,0.070729
